# NQ100 売買ルールバックテスト

Google Drive上のCSVを読み込み、東京時間の市場状態AIと売買シミュレーションを検証するためのColabノートブックです。

In [25]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess
import sys
from pathlib import Path

# ColabでGitHubから開いた場合、必要に応じてリポジトリをcloneします。
REPO_DIR = Path('/content/NQ100')
if not REPO_DIR.exists():
    subprocess.run(
        ['git', 'clone', 'https://github.com/hon-daisuki/NQ100.git', str(REPO_DIR)],
        check=True,
    )

sys.path.insert(0, str(REPO_DIR))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [26]:
import pandas as pd
import numpy as np

from src.labels import add_future_return_label, add_direction_label
from src.backtest import build_long_short_returns, summarize_returns
from src.metrics import describe_time_range, missing_summary

## データ読み込み

In [27]:
DATA_ROOT = Path('/content/drive')
CSV_CANDIDATES = [
    'USTEC_features_all.csv',
    'USTEC_1hour_features.csv',
]
PREFERRED_DIRS = [
    Path('/content/drive/MyDrive/CFD機械学習/backtest_ready'),
    Path('/content/drive/MyDrive/CFD機械学習'),
    Path('/content/drive/MyDrive'),
    DATA_ROOT,
]

def find_features_csv():
    checked = []
    for folder in PREFERRED_DIRS:
        for name in CSV_CANDIDATES:
            candidate = folder / name
            checked.append(str(candidate))
            if candidate.exists():
                return candidate

    matches = []
    for name in CSV_CANDIDATES:
        matches.extend(DATA_ROOT.rglob(name))
    if matches:
        return sorted(matches, key=lambda x: len(str(x)))[0]

    raise FileNotFoundError(
        '特徴量CSVが見つかりません。Driveに対象CSVがあるか、共有フォルダをMyDriveにショートカット追加してください。\n'
        + '探した候補:\n' + '\n'.join(checked)
    )

csv_path = find_features_csv()
print(f'Using CSV: {csv_path}')

df = pd.read_csv(csv_path)
df['日時'] = pd.to_datetime(df['日時'])
df = df.sort_values('日時').reset_index(drop=True)

describe_time_range(df)

Using CSV: /content/drive/MyDrive/CFD機械学習/NQ100/USTEC_1hour_features.csv


{'rows': 85831,
 'start': Timestamp('2011-10-01 00:00:00'),
 'end': Timestamp('2026-05-23 05:00:00'),
 'columns': ['日時',
  'open',
  'high',
  'low',
  'close',
  'return_1h',
  'range',
  'body',
  'upper_wick',
  'lower_wick',
  'hour',
  'dayofweek',
  'month',
  'sma_5',
  'sma_20',
  'ema_12',
  'ema_26',
  'macd',
  'macd_signal',
  'macd_hist',
  'bb_mid',
  'bb_std',
  'bb_upper',
  'bb_lower',
  'bb_width',
  'rsi_14',
  'atr_14']}

In [28]:
missing_summary(df)

,missing,missing_rate
bb_mid,19,0.000221
sma_20,19,0.000221
bb_lower,19,0.000221
bb_width,19,0.000221
bb_std,19,0.000221
bb_upper,19,0.000221
rsi_14,14,0.000163
atr_14,13,0.000151
sma_5,4,0.000047
return_1h,1,0.000012


## 東京時間フィルタとラベル作成

In [29]:
df['hour'] = df['日時'].dt.hour
df_tokyo = df[(df['hour'] >= 8) & (df['hour'] <= 12)].copy()

df_tokyo = add_future_return_label(df_tokyo, horizon=4, threshold=0.005)
df_tokyo = add_direction_label(df_tokyo, return_col='future_return_4h', threshold=0.005)

df_tokyo[['日時', 'close', 'future_return_4h', 'is_trend', 'signal']].tail()

,日時,close,future_return_4h,is_trend,signal
85809,2026-05-22 08:00:00,29549.9,0.000741,0,0
85810,2026-05-22 09:00:00,29564.5,NaN,0,0
85811,2026-05-22 10:00:00,29528.8,NaN,0,0
85812,2026-05-22 11:00:00,29548.1,NaN,0,0
85813,2026-05-22 12:00:00,29571.8,NaN,0,0


## 学習データ

In [30]:
features_trend = [
    'return_1h', 'range', 'body', 'upper_wick', 'lower_wick',
    'hour', 'dayofweek', 'month', 'sma_5', 'sma_20',
    'macd', 'macd_signal', 'macd_hist', 'bb_width', 'rsi_14', 'atr_14'
]

df_model = df_tokyo.dropna(subset=features_trend + ['is_trend', 'future_return_4h']).copy()
split_index = int(len(df_model) * 0.8)

X_train = df_model[features_trend].iloc[:split_index]
X_test = df_model[features_trend].iloc[split_index:]
y_train = df_model['is_trend'].iloc[:split_index]
y_test = df_model['is_trend'].iloc[split_index:]

len(X_train), len(X_test), y_train.mean(), y_test.mean()

(15000, 3751, np.float64(0.4706), np.float64(0.49853372434017595))

## LightGBM学習

In [31]:
!pip -q install lightgbm

from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

model_trend = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.03,
    num_leaves=31,
    class_weight='balanced',
    random_state=42,
)

model_trend.fit(X_train, y_train)
proba = model_trend.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)

print('Accuracy =', accuracy_score(y_test, pred))
print(confusion_matrix(y_test, pred))
print(classification_report(y_test, pred))

[LightGBM] [Info] Number of positive: 7059, number of negative: 7941
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001392 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3339
[LightGBM] [Info] Number of data points in the train set: 15000, number of used features: 16
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
Accuracy = 0.6475606504932018
[[ 996  885]
 [ 437 1433]]
              precision    recall  f1-score   support

           0       0.70      0.53      0.60      1881
           1       0.62      0.77      0.68      1870

    accuracy                           0.65      3751
   macro avg       0.66      0.65      0.64      3751
weighted avg       0.66      0.65      0.64      3751



## Direction Model

Predict whether the future 4-hour move is down, flat, or up.

In [32]:
direction_threshold = 0.005

df_model['direction_signal'] = 0
df_model.loc[df_model['future_return_4h'] > direction_threshold, 'direction_signal'] = 1
df_model.loc[df_model['future_return_4h'] < -direction_threshold, 'direction_signal'] = -1

# Convert -1/0/1 signals to 0/1/2 classes for LightGBM multiclass training.
signal_to_class = {-1: 0, 0: 1, 1: 2}
class_to_signal = {0: -1, 1: 0, 2: 1}

y_dir = df_model['direction_signal'].map(signal_to_class)
y_dir_train = y_dir.iloc[:split_index]
y_dir_test = y_dir.iloc[split_index:]

df_model['direction_signal'].value_counts(normalize=True).sort_index()


,proportion
direction_signal,
-1,0.204096
0,0.523812
1,0.272092


In [33]:
model_dir = LGBMClassifier(
    objective='multiclass',
    num_class=3,
    n_estimators=500,
    learning_rate=0.03,
    num_leaves=31,
    class_weight='balanced',
    random_state=42,
)

model_dir.fit(X_train, y_dir_train)
dir_proba = model_dir.predict_proba(X_test)
dir_class_pred = dir_proba.argmax(axis=1)
dir_signal_pred = pd.Series(dir_class_pred, index=X_test.index).map(class_to_signal)

print('Direction Accuracy =', accuracy_score(y_dir_test, dir_class_pred))
print(confusion_matrix(y_dir_test, dir_class_pred))
print(classification_report(
    y_dir_test,
    dir_class_pred,
    target_names=['down', 'flat', 'up'],
))


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001416 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3339
[LightGBM] [Info] Number of data points in the train set: 15000, number of used features: 16
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
Direction Accuracy = 0.4209544121567582
[[565 174  60]
 [867 935  79]
 [737 255  79]]
              precision    recall  f1-score   support

        down       0.26      0.71      0.38       799
        flat       0.69      0.50      0.58      1881
          up       0.36      0.07      0.12      1071

    accuracy                           0.42      3751
   macro avg       0.44      0.43      0.36      3751
weighted avg       0.50      0.42      0.41      3751



## Probability-Difference Signals

Use the gap between up probability and down probability instead of taking the largest class directly. This reduces one-sided bias and keeps weak predictions out of the market.

In [34]:
bt = df_model.iloc[split_index:].copy()
bt['trend_proba'] = proba
bt['down_proba'] = dir_proba[:, signal_to_class[-1]]
bt['flat_proba'] = dir_proba[:, signal_to_class[0]]
bt['up_proba'] = dir_proba[:, signal_to_class[1]]
bt['direction_edge'] = bt['up_proba'] - bt['down_proba']
bt['dir_confidence'] = np.maximum(bt['up_proba'], bt['down_proba'])

def make_probability_edge_signal(
    frame,
    trend_threshold=0.60,
    edge_threshold=0.20,
    min_direction_confidence=0.45,
    allow_long=True,
    allow_short=True,
):
    signal = pd.Series(0, index=frame.index, dtype=int)
    active = (
        (frame['trend_proba'] >= trend_threshold)
        & (frame['dir_confidence'] >= min_direction_confidence)
    )
    if allow_long:
        signal.loc[active & (frame['direction_edge'] >= edge_threshold)] = 1
    if allow_short:
        signal.loc[active & (frame['direction_edge'] <= -edge_threshold)] = -1
    return signal

bt[['trend_proba', 'down_proba', 'flat_proba', 'up_proba', 'direction_edge', 'dir_confidence']].describe()


,trend_proba,down_proba,flat_proba,up_proba,direction_edge,dir_confidence
count,3751.000000,3751.000000,3751.000000,3751.000000,3751.000000,3751.000000
mean,0.523614,0.482886,0.393803,0.123311,-0.359576,0.503656
std,0.262037,0.271680,0.281842,0.133039,0.321845,0.260993
min,0.002202,0.001586,0.015536,0.001145,-0.947354,0.001586
25%,0.356545,0.239265,0.159861,0.039927,-0.631974,0.301075
50%,0.585126,0.507436,0.304137,0.079736,-0.372186,0.531893
75%,0.735139,0.721183,0.581208,0.152392,-0.101819,0.722093
max,0.962369,0.963593,0.996877,0.831053,0.774986,0.963593


## Strategy Comparison

Compare long-only, short-only, and long-short variants across several edge thresholds.

In [35]:
strategy_rows = []
edge_thresholds = [0.10, 0.15, 0.20, 0.25, 0.30, 0.35]

for edge_threshold in edge_thresholds:
    for name, allow_long, allow_short in [
        ('long_short', True, True),
        ('long_only', True, False),
        ('short_only', False, True),
    ]:
        signal = make_probability_edge_signal(
            bt,
            trend_threshold=0.60,
            edge_threshold=edge_threshold,
            min_direction_confidence=0.45,
            allow_long=allow_long,
            allow_short=allow_short,
        )
        returns = build_long_short_returns(
            bt.assign(trade_signal=signal),
            signal_col='trade_signal',
            return_col='future_return_4h',
            cost_per_trade=0.0002,
        )
        summary = summarize_returns(returns)
        strategy_rows.append({
            'strategy': name,
            'edge_threshold': edge_threshold,
            'long_count': int((signal == 1).sum()),
            'short_count': int((signal == -1).sum()),
            'flat_count': int((signal == 0).sum()),
            **summary,
        })

strategy_results = pd.DataFrame(strategy_rows)
strategy_results.sort_values('total_return', ascending=False)


,strategy,edge_threshold,long_count,short_count,flat_count,trades,total_return,win_rate,mean_return,max_drawdown
1,long_only,0.10,124,0,3627,179.0,0.132087,0.018662,0.000035,-0.143748
7,long_only,0.20,106,0,3645,158.0,0.113583,0.015996,0.000030,-0.128240
13,long_only,0.30,76,0,3675,119.0,0.099375,0.011730,0.000026,-0.090953
4,long_only,0.15,111,0,3640,164.0,0.094456,0.016529,0.000025,-0.128240
10,long_only,0.25,92,0,3659,140.0,0.066579,0.013596,0.000018,-0.128240
16,long_only,0.35,58,0,3693,92.0,0.045415,0.008264,0.000012,-0.072359
15,long_short,0.35,58,1318,2375,1902.0,-0.564250,0.170621,-0.000194,-0.598916
0,long_short,0.10,124,1456,2171,2164.0,-0.581833,0.197281,-0.000201,-0.628045
17,short_only,0.35,0,1318,2433,1810.0,-0.583180,0.162357,-0.000206,-0.614082
3,long_short,0.15,111,1447,2193,2138.0,-0.591985,0.194348,-0.000207,-0.633938


## Selected Backtest

Start from the best row in the comparison table, then inspect recent trades. This is still a research result, not a deployable strategy.

In [36]:
best = strategy_results.sort_values('total_return', ascending=False).iloc[0]
selected_signal = make_probability_edge_signal(
    bt,
    trend_threshold=0.60,
    edge_threshold=float(best['edge_threshold']),
    min_direction_confidence=0.45,
    allow_long=best['strategy'] in ['long_short', 'long_only'],
    allow_short=best['strategy'] in ['long_short', 'short_only'],
)
bt['trade_signal'] = selected_signal
selected_returns = build_long_short_returns(
    bt,
    signal_col='trade_signal',
    return_col='future_return_4h',
    cost_per_trade=0.0002,
)

print('Selected strategy:')
print(best)
print('\nSignal counts:')
print(bt['trade_signal'].value_counts().sort_index())
print('\nBacktest summary:')
summarize_returns(selected_returns)


Selected strategy:
strategy          long_only
edge_threshold          0.1
long_count              124
short_count               0
flat_count             3627
trades                179.0
total_return       0.132087
win_rate           0.018662
mean_return        0.000035
max_drawdown      -0.143748
Name: 1, dtype: object

Signal counts:
trade_signal
0    3627
1     124
Name: count, dtype: int64

Backtest summary:


{'trades': 179.0,
 'total_return': 0.13208673472458465,
 'win_rate': 0.018661690215942415,
 'mean_return': 3.466461907697927e-05,
 'max_drawdown': -0.14374760791397756}

In [37]:
datetime_col = df.columns[0]
bt[[datetime_col, 'close', 'future_return_4h', 'trend_proba', 'down_proba', 'up_proba', 'direction_edge', 'trade_signal']].tail(30)


,日時,close,future_return_4h,trend_proba,down_proba,up_proba,direction_edge,trade_signal
85672,2026-05-14 09:00:00,29680.3,0.000229,0.727779,0.918980,0.030400,-0.888580,0
85673,2026-05-14 10:00:00,29607.1,0.002591,0.700447,0.914222,0.023610,-0.890611,0
85674,2026-05-14 11:00:00,29586.4,-0.002366,0.584758,0.922479,0.014065,-0.908413,0
85675,2026-05-14 12:00:00,29534.3,0.001534,0.464711,0.864653,0.022658,-0.841995,0
85694,2026-05-15 08:00:00,29687.1,-0.004180,0.199185,0.263019,0.033778,-0.229240,0
85695,2026-05-15 09:00:00,29683.8,-0.019068,0.816860,0.902286,0.021556,-0.880730,0
85696,2026-05-15 10:00:00,29516.4,-0.016374,0.849254,0.788836,0.073715,-0.715120,0
85697,2026-05-15 11:00:00,29579.6,-0.018874,0.863427,0.867692,0.040772,-0.826921,0
85698,2026-05-15 12:00:00,29563.0,-0.018736,0.823590,0.701283,0.105643,-0.595640,0
85717,2026-05-18 08:00:00,29117.8,-0.002160,0.614716,0.633982,0.084501,-0.549482,0
